# Chapter 2 &mdash; Slippery Roads: Definitions That Look the Same

**Concept 16 of the Chapter 2 decomposition:** *Slippery Roads: Telling Look-Alike Language Definitions Apart*

The chapter's designated trap: a <b>shared</b> exponent inside one set-builder is not the same as two <b>independent</b> ones across a concatenation.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter2-Lang/Concept-Slippery-Roads/Concept-Slippery-Roads.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.LangDef        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


The decisive case: is $\{0^n1^n : n\ge0\}$ the same as $\{0^n\}\{1^n\}$?

**No.** Inside a single set-builder the $n$ is **one shared variable**. Across a
concatenation the two exponents are **chosen independently**, so the second contains
$0^31^7$.

A language is determined by its **string membership**, never by the shape of the
notation used to describe it.

## 2. Definitions

### Two definitions that look alike

In [ ]:
B = 6
Eq01  = {'0'*n + '1'*n for n in range(B)}                      # shared n
Zeros = {'0'*n for n in range(B)}
Ones  = {'1'*n for n in range(B)}
Cat   = lcat(Zeros, Ones)                                      # independent exponents

### Even- and odd-length helpers

In [ ]:
LE = {'0'*(2*i)     for i in range(B)}      # even length
LO = {'0'*(2*i + 1) for i in range(B)}      # odd length

<!-- nav-strip -->

---

&larr;&nbsp;[Ch2&nbsp;15.&nbsp;Star Previewed as a Finite Union of Exponents](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter2-Lang/Concept-Star-Previewed/Concept-Star-Previewed.ipynb) &nbsp;&middot;&nbsp; [**Chapter 2** index](https://github.com/ganeshutah/Jove/blob/master/Chapter2-Lang/README.md) &nbsp;&middot;&nbsp; [Ch3&nbsp;1.&nbsp;Star: Three Equivalent Definitions](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter3-Star/Concept-Star-Three-Definitions/Concept-Star-Three-Definitions.ipynb)&nbsp;&rarr;

---

## 3. Tests

They are **not** equal, and the witness is concrete.

In [ ]:
print("Eq01 size :", len(Eq01), "   Cat size :", len(Cat))
extra = sorted(lminus(Cat, Eq01), key=len)[:6]
print("in Cat but NOT in Eq01 :", extra)
assert '0001111' in Cat or extra
assert Cat != Eq01
print()
print("'0^3 1^7' style strings live in Cat only. One shared n vs two free ones.")

But $L_7 = \{0^i1^j : i,j \ge 0\}$ **does** equal $\{0^i\}\{1^i\}$ &mdash; because there the
exponents were already independent.

In [ ]:
L7 = {'0'*i + '1'*j for i in range(B) for j in range(B)}
print("L7 == Cat ?", L7 == Cat)
assert L7 == Cat
print("  Yes -- both let the two counts vary freely.")

Even/odd: $\{0\}^* = L_E \cup L_O$.

In [ ]:
allzeros = lstar({'0'}, 9)
u = lunion(LE, LO)
print("L_E u L_O covers {0}* up to length 9 ?", lissubset({s for s in allzeros if len(s)<=9}, u))
print("L_E sample :", sorted(LE, key=len)[:4])
print("L_O sample :", sorted(LO, key=len)[:4])

The complement trap: $\overline{L_6}$ is **not** $\{0^i1^j : i \ne j\}$.

In [ ]:
Sigma_star_4 = lstar({'0','1'}, 4)
L6           = {s for s in Sigma_star_4 if s == '0'*s.count('0') + '1'*s.count('1')
                                        and s.count('0') == s.count('1')}
L8           = {s for s in Sigma_star_4 if s == '0'*s.count('0') + '1'*s.count('1')
                                        and s.count('0') != s.count('1')}
notL6        = lminus(Sigma_star_4, L6)
missed       = sorted(lminus(notL6, L8), key=len)[:6]
print("strings in complement(L6) but NOT in L8 :", missed)
assert missed, "there must be strings that are not even of the form 0^i 1^j"
print()
print("Those are not of the form 0^i 1^j AT ALL -- e.g. '10', '0101'.")
print("The complement is taken over ALL of Sigma*, not just the nice-shaped strings.")

## 4. Exercises


1. Show $L_E = \{0^{2i}\} = \{(00)^i\}$ by testing both to length 10.
2. Which of $L_1 \ldots L_6$ in the book equal $Eq_{01}$? Test each.
3. List four strings in $\overline{L_6}$ that are not in $L_8$, and describe the
   class they form in English.

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter2-Lang/Concept-Slippery-Roads')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')